In [2]:
# =========================
# PNAD COVID - BAIXAR + EXTRAIR + LER + CONCAT + mes_ref
# (copie e cole tudo)
# =========================

import os
import zipfile
import requests
import pandas as pd
from glob import glob

# =========================
# 1) CONFIGURAÇÕES
# =========================
PASTA_DADOS = "dados_pnad"
os.makedirs(PASTA_DADOS, exist_ok=True)

# >>> Ajuste aqui os 3 meses do seu trabalho (formato "MMAAAA")
meses = ["082020", "092020", "102020"]

BASE_URL = (
    "https://ftp.ibge.gov.br/Trabalho_e_Rendimento/"
    "Pesquisa_Nacional_por_Amostra_de_Domicilios_PNAD_COVID19/"
    "Microdados/Dados/"
)

# =========================
# 2) FUNÇÕES
# =========================
def baixar_zip(mmaaaa: str) -> str:
    nome_zip = f"PNAD_COVID_{mmaaaa}.zip"
    url = BASE_URL + nome_zip
    caminho_zip = os.path.join(PASTA_DADOS, nome_zip)

    if os.path.exists(caminho_zip):
        print(f"[OK] Já existe: {caminho_zip}")
        return caminho_zip

    print(f"[BAIXANDO] {url}")
    resp = requests.get(url, timeout=240)
    resp.raise_for_status()

    with open(caminho_zip, "wb") as f:
        f.write(resp.content)

    print(f"[OK] Baixado: {caminho_zip}")
    return caminho_zip


def extrair_zip(caminho_zip: str) -> str:
    nome_zip = os.path.basename(caminho_zip)  # PNAD_COVID_092020.zip
    mmaaaa = nome_zip.replace("PNAD_COVID_", "").replace(".zip", "")
    pasta_mes = os.path.join(PASTA_DADOS, f"PNAD_COVID_{mmaaaa}")
    os.makedirs(pasta_mes, exist_ok=True)

    # se já tiver arquivos extraídos, não reextrai
    if len(os.listdir(pasta_mes)) > 0:
        print(f"[OK] Já extraído: {pasta_mes}")
        return pasta_mes

    print(f"[EXTRAINDO] {caminho_zip} -> {pasta_mes}")
    with zipfile.ZipFile(caminho_zip, "r") as z:
        z.extractall(pasta_mes)

    print(f"[OK] Extraído: {pasta_mes}")
    return pasta_mes


def extrair_mes_ref(nome_pasta: str) -> str:
    # 'PNAD_COVID_092020' -> '2020-09'
    mm_aaaa = nome_pasta.split("_")[-1]  # 092020
    mm = mm_aaaa[:2]
    aaaa = mm_aaaa[2:]
    return f"{aaaa}-{mm}"


def encontrar_arquivo_microdados(pasta_mes: str) -> str:
    # procura CSV/TXT e pega o maior (normalmente é o microdado principal)
    candidatos = []
    for ext in ("*.csv", "*.CSV", "*.txt", "*.TXT"):
        candidatos.extend(glob(os.path.join(pasta_mes, ext)))

    if not candidatos:
        for ext in ("**/*.csv", "**/*.CSV", "**/*.txt", "**/*.TXT"):
            candidatos.extend(glob(os.path.join(pasta_mes, ext), recursive=True))

    if not candidatos:
        raise FileNotFoundError(f"Nenhum CSV/TXT encontrado em: {pasta_mes}")

    return max(candidatos, key=os.path.getsize)


def detectar_sep(caminho: str) -> str:
    # lê um pedaço do arquivo e decide entre ',' e ';'
    with open(caminho, "r", encoding="latin1", errors="ignore") as f:
        amostra = f.read(20000)
    return "," if amostra.count(",") >= amostra.count(";") else ";"


def ler_microdados(caminho: str) -> pd.DataFrame:
    sep = detectar_sep(caminho)
    print(f"    -> separador detectado: '{sep}'")
    # latin1 costuma funcionar bem em base BR com acentos
    return pd.read_csv(caminho, sep=sep, encoding="latin1", low_memory=False)

# =========================
# 3) PIPELINE
# =========================
pastas_extraidas = []

# 3.1 baixar + extrair
for m in meses:
    zip_path = baixar_zip(m)
    pasta_mes = extrair_zip(zip_path)
    pastas_extraidas.append(pasta_mes)

# 3.2 ler + mes_ref + concat
dfs = []

for pasta_mes in sorted(pastas_extraidas):
    nome_pasta = os.path.basename(pasta_mes)      # PNAD_COVID_092020
    mes_ref = extrair_mes_ref(nome_pasta)         # 2020-09

    arquivo = encontrar_arquivo_microdados(pasta_mes)
    print(f"[LENDO] {mes_ref} -> {arquivo}")

    df_mes = ler_microdados(arquivo)
    df_mes["mes_ref"] = mes_ref
    dfs.append(df_mes)

df_all = pd.concat(dfs, ignore_index=True)

print("\n===== RESULTADO FINAL =====")
print("Shape (linhas, colunas):", df_all.shape)
print("\nRegistros por mês:")
print(df_all["mes_ref"].value_counts().sort_index())

print("\nPrimeiras 25 colunas:")
print(list(df_all.columns[:25]))

print("\nAmostra:")
print(df_all.head(3))

# (Opcional) salvar
# df_all.to_csv("pnad_covid_3meses.csv", index=False)
# df_all.to_parquet("pnad_covid_3meses.parquet", index=False)


[OK] Já existe: dados_pnad\PNAD_COVID_082020.zip
[OK] Já extraído: dados_pnad\PNAD_COVID_082020
[OK] Já existe: dados_pnad\PNAD_COVID_092020.zip
[OK] Já extraído: dados_pnad\PNAD_COVID_092020
[OK] Já existe: dados_pnad\PNAD_COVID_102020.zip
[OK] Já extraído: dados_pnad\PNAD_COVID_102020
[LENDO] 2020-08 -> dados_pnad\PNAD_COVID_082020\PNAD_COVID_082020.csv
    -> separador detectado: ','
[LENDO] 2020-09 -> dados_pnad\PNAD_COVID_092020\PNAD_COVID_092020.csv
    -> separador detectado: ','
[LENDO] 2020-10 -> dados_pnad\PNAD_COVID_102020\PNAD_COVID_102020.csv
    -> separador detectado: ','

===== RESULTADO FINAL =====
Shape (linhas, colunas): (1154279, 146)

Registros por mês:
mes_ref
2020-08    386520
2020-09    387298
2020-10    380461
Name: count, dtype: int64

Primeiras 25 colunas:
['Ano', 'UF', 'CAPITAL', 'RM_RIDE', 'V1008', 'V1012', 'V1013', 'V1016', 'Estrato', 'UPA', 'V1022', 'V1023', 'V1030', 'V1031', 'V1032', 'posest', 'A001', 'A001A', 'A001B1', 'A001B2', 'A001B3', 'A002', 'A003'

In [9]:
import pandas as pd
import numpy as np

# =========================================================
# 0) PRÉ-REQUISITO
# df_all já existe (o seu df com 3 meses + mes_ref)
# e você já gerou mapa_variaveis.xlsx
# =========================================================

# =========================================================
# 1) CARREGAR MAPA (ou usar o "mapa" se já estiver em memória)
# =========================================================
try:
    mapa = pd.read_excel("mapa_variaveis.xlsx")  # precisa openpyxl
except Exception:
    # se não der, assume que você ainda tem "mapa" em memória
    pass

# completar descrições básicas que ficaram NaN
complemento = {
    "Ano": "Ano da pesquisa",
    "UF": "Unidade da Federação",
    "CAPITAL": "Indicador de capital",
    "RM_RIDE": "Indicador RM/RIDE (região metropolitana / ride)",
    "Estrato": "Estrato amostral",
    "UPA": "Unidade Primária de Amostragem",
    "posest": "Pós-estratificação (ajuste amostral)",
    "mes_ref": "Mês de referência (criado no ETL)",
}
if "descricao" in mapa.columns:
    mapa["descricao"] = mapa.apply(
        lambda r: complemento.get(r["coluna"], r["descricao"]) if pd.isna(r["descricao"]) else r["descricao"],
        axis=1
    )

# =========================================================
# 2) AJUDAS: LISTAR CANDIDATAS POR TEMA (SEM VOCÊ TER QUE ADIVINHAR)
# =========================================================
def mostrar_candidatas(palavras, top=40):
    if "descricao" not in mapa.columns:
        print("Mapa não tem coluna 'descricao'.")
        return
    pat = "|".join(palavras)
    cand = mapa[mapa["descricao"].str.contains(pat, case=False, na=False)][["coluna","prefixo","descricao"]]
    print(cand.head(top).to_string(index=False))

print("\n=== CANDIDATAS (SINTOMAS) ===")
mostrar_candidatas(["febre","tosse","garganta","respirar","olfato","sabor","fadiga","cabeça","muscular","peito","náusea","nariz"], top=60)

print("\n=== CANDIDATAS (COMPORTAMENTO / ATENDIMENTO) ===")
mostrar_candidatas(["estabelecimento","atendimento","posto","pronto","hospital","intern", "plano de saúde"], top=80)

print("\n=== CANDIDATAS (ECONÔMICO / TRABALHO / RENDA) ===")
mostrar_candidatas(["trabalho","ocup", "renda", "rendimento", "salário", "auxílio", "benefício", "desempreg"], top=80)

print("\n=== TODAS AS VARIÁVEIS A (CARACTERÍSTICAS DA POPULAÇÃO) ===")
print(mapa[mapa["prefixo"]=="A"][["coluna","descricao"]].to_string(index=False))

# =========================================================
# 3) LISTA SUGERIDA DE 20 VARIÁVEIS (VOCÊ PODE EDITAR)
#    - 8 sintomas
#    - 6 comportamento/saúde
#    - 4 população (A*)
#    - 2 econômico (você vai escolher olhando as candidatas)
# =========================================================
# Sintomas (8)
vars_sintomas = ["B0011","B0012","B0013","B0014","B0015","B0019","B00111","B00112"]

# Comportamento/saúde (6)
vars_saude = ["B002","B0043","B0046","B005","B006","B007"]

# População (4) -> CONFIRA NO PRINT de A* acima quais você quer usar
# (A002..A005 aparecem no seu df, mas confirme a descrição certinha no Excel)
vars_pop = ["A002","A003","A004","A005"]

# Econômico (2) -> ESCOLHA AQUI depois de olhar as candidatas econômicas
# Exemplo placeholders (troque por variáveis reais do seu mapa)
vars_econ = ["C001", "D001"]  # <<< TROCAR para códigos econômicos existentes no seu df/mapa

# Monta a lista final (no máximo 20)
cols_20 = vars_sintomas + vars_saude + vars_pop + vars_econ

# Garantir que só entra o que existe no df_all
cols_20 = [c for c in cols_20 if c in df_all.columns]

print("\n=== COLUNAS SELECIONADAS (existentes no df_all) ===")
print(cols_20)
print("Qtd:", len(cols_20))

# =========================================================
# 4) CRIAR df_20 (com mes_ref + UF para cortes)
# =========================================================
base_cols = ["mes_ref","UF"]
base_cols = [c for c in base_cols if c in df_all.columns]

df_20 = df_all[base_cols + cols_20].copy()

# =========================================================
# 5) FUNÇÕES DE LIMPEZA SIM/NÃO (1=Sim, 2=Não)
# =========================================================
def to_yes_no(series):
    # retorna 1 para sim, 0 para não, NaN para outros
    return np.where(series == 1, 1,
           np.where(series == 2, 0, np.nan))

# aplica em sintomas e variáveis de saúde que são sim/não
binarias = vars_sintomas + vars_saude
binarias = [c for c in binarias if c in df_20.columns]

for c in binarias:
    df_20[c] = to_yes_no(df_20[c])

# =========================================================
# 6) INDICADORES PRINCIPAIS (POR MÊS)
# =========================================================
# 6.1 "Teve pelo menos 1 sintoma"
sintomas_ok = [c for c in vars_sintomas if c in df_20.columns]
if sintomas_ok:
    df_20["teve_algum_sintoma"] = (df_20[sintomas_ok].sum(axis=1, min_count=1) > 0).astype("float")
else:
    df_20["teve_algum_sintoma"] = np.nan

# 6.2 resumo por mês
resumo_mes = df_20.groupby("mes_ref").agg(
    n=("mes_ref","size"),
    taxa_algum_sintoma=("teve_algum_sintoma","mean"),
    febre=("B0011","mean") if "B0011" in df_20.columns else ("teve_algum_sintoma","mean"),
    tosse=("B0012","mean") if "B0012" in df_20.columns else ("teve_algum_sintoma","mean"),
    falta_ar=("B0014","mean") if "B0014" in df_20.columns else ("teve_algum_sintoma","mean"),
    perda_olfato_sabor=("B00111","mean") if "B00111" in df_20.columns else ("teve_algum_sintoma","mean"),
    buscou_estabelecimento=("B002","mean") if "B002" in df_20.columns else ("teve_algum_sintoma","mean"),
    internacao=("B005","mean") if "B005" in df_20.columns else ("teve_algum_sintoma","mean"),
    plano_saude=("B007","mean") if "B007" in df_20.columns else ("teve_algum_sintoma","mean"),
).reset_index()

# transforma em %
pct_cols = [c for c in resumo_mes.columns if c not in ["mes_ref","n"]]
resumo_mes[pct_cols] = (resumo_mes[pct_cols] * 100).round(2)

print("\n=== RESUMO POR MÊS (%) ===")
print(resumo_mes.to_string(index=False))

# =========================================================
# 7) (OPCIONAL) CORTE POR UF (ex.: top 10 UFs por taxa de algum sintoma)
# =========================================================
if "UF" in df_20.columns:
    resumo_uf = (
        df_20.groupby("UF")["teve_algum_sintoma"]
        .mean()
        .sort_values(ascending=False)
        .head(10) * 100
    ).round(2)
    print("\n=== TOP 10 UFs por taxa de algum sintoma (%) ===")
    print(resumo_uf)

# =========================================================
# 8) SALVAR df_20 PARA USAR NO POWER BI / SUBIR NO BANCO
# =========================================================
# df_20.to_csv("pnad_covid_20vars_3meses.csv", index=False, encoding="utf-8-sig")
# df_20.to_parquet("pnad_covid_20vars_3meses.parquet", index=False)



=== CANDIDATAS (SINTOMAS) ===
coluna prefixo                                            descricao
 B0011       B                        Na semana passada teve febre?
B00111       B     Na semana passada teve perda de cheiro ou sabor?
B00112       B                 Na semana passada teve dor muscular?
 B0012       B                        Na semana passada teve tosse?
 B0013       B              Na semana passada teve dor de garganta?
 B0014       B    Na semana passada teve dificuldade para respirar?
 B0015       B                Na semana passada teve dor de cabeça?
 B0016       B                 Na semana passada teve dor no peito?
 B0017       B                       Na semana passada teve náusea?
 B0018       B Na semana passada teve nariz entupido ou escorrendo?
 B0019       B                       Na semana passada teve fadiga?

=== CANDIDATAS (COMPORTAMENTO / ATENDIMENTO) ===
coluna prefixo                                                                                         